# Capstone Project 1
# Multi-Agent Remittance Ingestion & Processing System
### Planner–Executor + Supervisor Pattern, with MCP-style Tool Contracts

**Maps to curriculum tracks:** A1 (Agent Architecture Patterns), A2 (Multi-Agent Orchestration —
LangGraph-style StateGraph), A3 (MCP & A2A as Integration Contracts), A6 (Functional Mapping:
business process → agents & HITL).

---

## 1. Problem Statement

A finance shared-services team receives **remittance advice** from four heterogeneous channels:

- Free-text customer **emails**
- **ERP exports** (CSV/flat files from D365 or similar)
- Scanned/converted **PDF remittance advices**
- Ad-hoc **EDI-like** structured text

Today this is triaged manually: an analyst reads each document, decides what type it is, extracts
customer, invoice references, amounts and dates, and keys them into the cash-application system.
This is slow, inconsistent, and does not scale with transaction volume.

**Goal:** Design and build a multi-agent system that:

1. Uses a **Supervisor agent** to classify the incoming source type and route it (Supervisor /
   Hierarchical pattern, A1).
2. Uses specialist **Executor agents**, each wrapping an extraction **tool** behind an
   **MCP-style contract** (fixed name, input schema, `run()` method) so tools can be swapped
   without touching agent logic (A3).
3. Runs a **Planner–Executor** loop: the planner decides the next step (extract → validate →
   finalize / escalate) based on the evolving state — the same idea LangGraph implements via a
   `StateGraph` with conditional edges (A2).
4. Escalates low-confidence extractions to a **human-in-the-loop (HITL)** review node instead of
   silently guessing (A6).
5. Produces a clean, structured, audited dataset ready for downstream **Payment Matching**
   (Capstone Project 2).

We simulate the LangGraph *API surface* (`add_node`, `add_edge`, `add_conditional_edges`,
`compile`, `invoke`) with a small dependency-free `StateGraph` class, so this notebook runs
anywhere. In a real Azure deployment you would swap this for the actual `langgraph` package —
the agent/tool code below is written so that swap only touches the orchestration wiring, not the
business logic (that separation is the point of A2/A3).


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import re
import json
import uuid
import random
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Callable, Dict, List, Any, Optional

import pandas as pd

random.seed(42)
pd.set_option("display.max_colwidth", 120)


## 2. Mini Orchestration Framework (LangGraph-style `StateGraph`)

This is a minimal, dependency-free stand-in for `langgraph.graph.StateGraph`. It supports:

- `add_node(name, fn)` — a node is any callable `state -> state`
- `add_edge(a, b)` — unconditional transition
- `add_conditional_edges(name, router_fn, mapping)` — branch based on state (ReAct-style decision)
- `compile()` → returns a runnable graph with `.invoke(state)`

This mirrors the real LangGraph API closely enough that migrating to the real library later is a
mechanical swap, which is exactly the kind of architecture decision covered in A1/A2.


In [ ]:
class StateGraph:
    """Dependency-free stand-in for langgraph.graph.StateGraph."""

    START = "__start__"
    END = "__end__"

    def __init__(self):
        self.nodes: Dict[str, Callable[[dict], dict]] = {}
        self.edges: Dict[str, str] = {}
        self.conditional_edges: Dict[str, Any] = {}
        self.entry_point: Optional[str] = None

    def add_node(self, name: str, fn: Callable[[dict], dict]):
        self.nodes[name] = fn
        return self

    def set_entry_point(self, name: str):
        self.entry_point = name
        return self

    def add_edge(self, a: str, b: str):
        self.edges[a] = b
        return self

    def add_conditional_edges(self, name: str, router_fn: Callable[[dict], str], mapping: Dict[str, str]):
        self.conditional_edges[name] = (router_fn, mapping)
        return self

    def compile(self):
        return CompiledGraph(self)


class CompiledGraph:
    def __init__(self, graph: StateGraph):
        self.graph = graph

    def invoke(self, state: dict, max_steps: int = 25) -> dict:
        current = self.graph.entry_point
        trace = state.setdefault("_trace", [])
        steps = 0
        while current != StateGraph.END and steps < max_steps:
            fn = self.graph.nodes[current]
            trace.append({"node": current, "ts": datetime.now().isoformat()})
            state = fn(state)
            if current in self.graph.conditional_edges:
                router_fn, mapping = self.graph.conditional_edges[current]
                branch = router_fn(state)
                current = mapping[branch]
            else:
                current = self.graph.edges.get(current, StateGraph.END)
            steps += 1
        return state


## 3. MCP-style Tool Contract (A3)

Every extraction capability is wrapped as a **tool** with a fixed contract: a `name`, a declared
`input_schema`, and a `run(payload) -> dict` method. Agents never call extraction logic directly —
they call `tool.run(...)`. This is the same discipline an MCP server enforces: a stable contract
that lets you swap the implementation (e.g. a regex parser today, an LLM-based extractor tomorrow)
without changing any agent code.


In [ ]:
class MCPTool:
    """Minimal MCP-style tool contract: name + input_schema + run()."""

    name: str = "base_tool"
    input_schema: Dict[str, str] = {}

    def run(self, payload: dict) -> dict:
        raise NotImplementedError


class EmailRemittanceExtractor(MCPTool):
    name = "extract_email_remittance"
    input_schema = {"raw_text": "string"}

    INVOICE_RE = re.compile(r"INV[-\s]?(\d{4,8})", re.I)
    AMOUNT_RE = re.compile(r"(?:USD|INR|\$|₹)\s?([\d,]+\.\d{2})")
    CUSTOMER_RE = re.compile(r"From:\s*(.+?)\s*<")

    def run(self, payload: dict) -> dict:
        text = payload["raw_text"]
        invoices = self.INVOICE_RE.findall(text)
        amounts = [float(a.replace(",", "")) for a in self.AMOUNT_RE.findall(text)]
        cust = self.CUSTOMER_RE.search(text)
        confidence = 0.95 if invoices and amounts else (0.55 if invoices or amounts else 0.15)
        return {
            "source_type": "email",
            "customer": cust.group(1).strip() if cust else "UNKNOWN",
            "invoice_refs": [f"INV-{i}" for i in invoices],
            "amount": sum(amounts) if amounts else None,
            "confidence": confidence,
        }


class ERPFlatFileExtractor(MCPTool):
    name = "extract_erp_row"
    input_schema = {"row": "dict"}

    REQUIRED = ["customer", "invoice_ref", "amount", "currency", "payment_date"]

    def run(self, payload: dict) -> dict:
        row = payload["row"]
        missing = [f for f in self.REQUIRED if not row.get(f)]
        valid = len(missing) == 0
        confidence = 0.98 if valid else 0.4
        return {
            "source_type": "erp_csv",
            "customer": row.get("customer", "UNKNOWN"),
            "invoice_refs": [row["invoice_ref"]] if row.get("invoice_ref") else [],
            "amount": row.get("amount"),
            "confidence": confidence,
            "validation_errors": missing,
        }


class PDFRemittanceExtractor(MCPTool):
    name = "extract_pdf_remittance"
    input_schema = {"pdf_text_lines": "list[string]"}

    LINE_RE = re.compile(r"(INV-\d+)\s+([A-Za-z ]+)\s+([\d,]+\.\d{2})")

    def run(self, payload: dict) -> dict:
        lines = payload["pdf_text_lines"]
        invoices, customers, amounts = [], set(), []
        for ln in lines:
            m = self.LINE_RE.search(ln)
            if m:
                invoices.append(m.group(1))
                customers.add(m.group(2).strip())
                amounts.append(float(m.group(3).replace(",", "")))
        confidence = 0.9 if invoices else 0.2
        return {
            "source_type": "pdf",
            "customer": next(iter(customers), "UNKNOWN"),
            "invoice_refs": invoices,
            "amount": round(sum(amounts), 2) if amounts else None,
            "confidence": confidence,
        }


## 4. Agents (Supervisor / Executors / Validator / HITL)

- **SupervisorAgent** — classifies source type and routes to the right executor (Supervisor
  pattern, A1).
- **ExecutorAgents** — one per channel, each a thin wrapper calling its MCP tool.
- **ValidationAgent** — checks the extracted record against business rules.
- **HITLAgent** — simulates a human reviewer resolving low-confidence cases (in production this
  node would pause the graph and wait for an external event / checkpoint, then resume — the
  planner–executor loop continues once the human input lands back in state).


In [ ]:
def supervisor_node(state: dict) -> dict:
    """Classifies the source and stamps the routing decision (ReAct-style: reason then act)."""
    src = state["input"]["channel"]
    state["route"] = src
    state["reasoning"] = f"Observed channel='{src}' -> routing to specialist extractor."
    return state


def email_executor_node(state: dict) -> dict:
    tool = EmailRemittanceExtractor()
    state["extraction"] = tool.run({"raw_text": state["input"]["payload"]})
    return state


def erp_executor_node(state: dict) -> dict:
    tool = ERPFlatFileExtractor()
    state["extraction"] = tool.run({"row": state["input"]["payload"]})
    return state


def pdf_executor_node(state: dict) -> dict:
    tool = PDFRemittanceExtractor()
    state["extraction"] = tool.run({"pdf_text_lines": state["input"]["payload"]})
    return state


def validation_node(state: dict) -> dict:
    ext = state["extraction"]
    errors = []
    if not ext.get("invoice_refs"):
        errors.append("no_invoice_ref_found")
    if ext.get("amount") in (None, 0):
        errors.append("no_amount_found")
    state["validation_errors"] = errors
    state["needs_hitl"] = bool(errors) or ext["confidence"] < 0.6
    return state


def hitl_node(state: dict) -> dict:
    """Simulated human review: in production this would pause the graph (checkpoint) and wait
    for a UI action. Here we deterministically 'fix' correctable cases and reject the rest, and
    we log the decision for audit."""
    ext = state["extraction"]
    if ext.get("amount") and ext.get("invoice_refs"):
        decision = "approved_with_edits"
        ext["confidence"] = 0.99
    else:
        decision = "rejected_insufficient_data"
    state["hitl_decision"] = decision
    return state


def finalize_node(state: dict) -> dict:
    ext = state["extraction"]
    state["result"] = {
        "record_id": str(uuid.uuid4())[:8],
        "source_type": ext["source_type"],
        "customer": ext["customer"],
        "invoice_refs": ext["invoice_refs"],
        "amount": ext["amount"],
        "confidence": ext["confidence"],
        "straight_through": not state.get("needs_hitl", False),
        "hitl_decision": state.get("hitl_decision"),
        "validation_errors": state.get("validation_errors", []),
    }
    return state


def route_by_channel(state: dict) -> str:
    return state["route"]


def route_by_confidence(state: dict) -> str:
    return "hitl" if state["needs_hitl"] else "skip"


In [ ]:
# ── Build & compile the graph ────────────────────────────────────────────────
g = StateGraph()
g.add_node("supervisor", supervisor_node)
g.add_node("email_exec", email_executor_node)
g.add_node("erp_exec", erp_executor_node)
g.add_node("pdf_exec", pdf_executor_node)
g.add_node("validate", validation_node)
g.add_node("hitl", hitl_node)
g.add_node("finalize", finalize_node)

g.set_entry_point("supervisor")
g.add_conditional_edges("supervisor", route_by_channel, {
    "email": "email_exec", "erp_csv": "erp_exec", "pdf": "pdf_exec",
})
g.add_edge("email_exec", "validate")
g.add_edge("erp_exec", "validate")
g.add_edge("pdf_exec", "validate")
g.add_conditional_edges("validate", route_by_confidence, {
    "hitl": "hitl", "skip": "finalize",
})
g.add_edge("hitl", "finalize")
g.add_edge("finalize", StateGraph.END)

remittance_graph = g.compile()
print("Graph compiled with nodes:", list(g.nodes.keys()))


## 5. Test Cases (minimum 5)

Each case pushes one raw document through `remittance_graph.invoke(state)` and inspects the
resulting audit trace and final record.


In [ ]:
# ── Case 1: Clean email remittance (straight-through expected) ──────────────
case1_input = {
    "channel": "email",
    "payload": (
        "From: Priya Shah <priya.shah@acme-corp.com>\n"
        "Subject: Remittance Advice\n\n"
        "Please find attached payment against INV-104233 and INV-104240.\n"
        "Total amount remitted: USD 12,450.00\n"
    ),
}
state1 = remittance_graph.invoke({"input": case1_input})
print(json.dumps(state1["result"], indent=2))


In [ ]:
# ── Case 2: Valid ERP CSV row (straight-through expected) ────────────────────
case2_input = {
    "channel": "erp_csv",
    "payload": {
        "customer": "Contoso Retail Ltd", "invoice_ref": "INV-500219",
        "amount": 8899.50, "currency": "USD", "payment_date": "2026-08-10",
    },
}
state2 = remittance_graph.invoke({"input": case2_input})
print(json.dumps(state2["result"], indent=2))


In [ ]:
# ── Case 3: PDF remittance advice, multi-line table ──────────────────────────
case3_input = {
    "channel": "pdf",
    "payload": [
        "Remittance Advice - Fabrikam Inc",
        "INV-900011 Fabrikam Inc 3,200.00",
        "INV-900012 Fabrikam Inc 1,750.25",
    ],
}
state3 = remittance_graph.invoke({"input": case3_input})
print(json.dumps(state3["result"], indent=2))


In [ ]:
# ── Case 4: Malformed ERP row (missing amount) -> validation fails -> HITL ───
case4_input = {
    "channel": "erp_csv",
    "payload": {
        "customer": "Northwind Traders", "invoice_ref": "INV-700044",
        "amount": None, "currency": "USD", "payment_date": "2026-08-11",
    },
}
state4 = remittance_graph.invoke({"input": case4_input})
print(json.dumps(state4["result"], indent=2))
print("\nTrace:", [t["node"] for t in state4["_trace"]])


In [ ]:
# ── Case 5: Ambiguous email, no invoice ref, no amount -> HITL rejects ───────
case5_input = {
    "channel": "email",
    "payload": "From: ops@unknownvendor.io <ops@unknownvendor.io>\nHi, payment sent, details to follow.",
}
state5 = remittance_graph.invoke({"input": case5_input})
print(json.dumps(state5["result"], indent=2))
print("\nTrace:", [t["node"] for t in state5["_trace"]])


### Bonus Case 6 — multi-source consolidation batch

Demonstrates the supervisor routing a mixed batch end to end, which is the actual daily operating
pattern (Day 1 use case: *Data Ingestion* + *Remittance Processing*).


In [ ]:
batch = [case1_input, case2_input, case3_input, case4_input, case5_input]
results = []
for item in batch:
    s = remittance_graph.invoke({"input": item})
    results.append(s["result"])

remittance_df = pd.DataFrame(results)
remittance_df


## 6. End Result & Conclusion

The graph consolidates four heterogeneous channels into one structured, audited table with a
`straight_through` flag and an `hitl_decision` trail for anything the system was not confident
about — exactly the behaviour the *Data Ingestion* and *Remittance Processing* use cases from the
curriculum ask for.


In [ ]:
straight_through_rate = remittance_df["straight_through"].mean()
print(f"Straight-through processing rate: {straight_through_rate:.0%}")
print(f"Records requiring HITL: {(~remittance_df['straight_through']).sum()} / {len(remittance_df)}")
remittance_df.groupby("source_type")["confidence"].mean().round(2)


**Conclusion**

- A **supervisor + specialist-executor** architecture cleanly separates *routing logic* from
  *extraction logic* — new channels (e.g. WhatsApp, EDI 820) are added as one new node + one new
  MCP tool, with zero change to the supervisor or validator.
- Wrapping every extractor behind an **MCP-style contract** means the regex-based extractors used
  here can be replaced by an LLM-based or a vendor OCR-based extractor later without touching
  agent orchestration code — this is the practical value of A3 in this curriculum.
- The **conditional edge on confidence** operationalises human-in-the-loop as a first-class graph
  node rather than an afterthought, satisfying the governance expectation that low-confidence
  automation must not silently proceed (foreshadows Day 3's Responsible AI module).
- Straight-through rate is a single, trackable KPI a shared-services leader can put on a
  dashboard — directly reusable in Capstone 3's reporting agent.

## 7. Applications

- **Accounts Receivable / Cash Application** — automatic ingestion of remittances ahead of
  payment matching (feeds Capstone 2 directly).
- **Treasury operations** — consolidating bank advices, SWIFT MT101/MT940 messages, and portal
  downloads into one canonical structure.
- **Accounts Payable** — vendor invoice intake automation using the same supervisor/executor
  pattern with different extractors.
- **Insurance claims intake** — heterogeneous claim documents (email, PDF, portal) routed to
  specialist extraction agents, with HITL for low-confidence claims.
- **Procurement** — PO acknowledgement and goods-receipt document ingestion at scale, feeding
  downstream 3-way match (Capstone 2, Case 2).
